## Corn Dataset Validation

Testing PLSR, SVR, RF and CNN on the publicly available NIR Corn dataset.
This validates that our model code is correct by comparing against published results.
The corn dataset has 80 samples with 700 spectral channels predicting protein content.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/priyanshu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Using device: cuda


## Load Corn Dataset

Load the corn NIR dataset. X contains NIR spectra (80 samples x 700 wavelengths).
y contains protein content labels for each sample.

In [2]:
import scipy.io

# Load corn dataset
mat = scipy.io.loadmat('corn.mat')

# Check what's inside
print("Keys in corn.mat:")
for key in mat.keys():
    if not key.startswith('_'):
        print(f"  {key}: {mat[key].shape if hasattr(mat[key], 'shape') else type(mat[key])}")

Keys in corn.mat:
  information: (18,)
  m5spec: (1, 1)
  mp5spec: (1, 1)
  mp6spec: (1, 1)
  propvals: (1, 1)
  m5nbs: (1, 1)
  mp5nbs: (1, 1)
  mp6nbs: (1, 1)


In [4]:
import scipy.io
import numpy as np

mat = scipy.io.loadmat('corn.mat')

# Extract spectra and labels
X_corn = mat['m5spec']['data'][0, 0]     # 80 x 700 NIR spectra
prop   = mat['propvals']['data'][0, 0]   # 80 x 4 (Moisture, Oil, Protein, Starch)

# Protein is column index 2
y_corn = prop[:, 2]

print(f"X_corn shape: {X_corn.shape}")
print(f"y_corn shape: {y_corn.shape}")
print(f"Protein range: {y_corn.min():.3f} to {y_corn.max():.3f} %")
print(f"Protein mean:  {y_corn.mean():.3f} %")

X_corn shape: (80, 700)
y_corn shape: (80,)
Protein range: 7.654 to 9.711 %
Protein mean:  8.668 %


In [5]:
from scipy.signal import savgol_filter

def savitzky_golay_smooth(X, window=11, poly=3):
    return savgol_filter(X, window_length=window, polyorder=poly, axis=1)

def snv(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / std

# Apply SG + SNV
X_corn_pre = snv(savitzky_golay_smooth(X_corn))
print(f"Preprocessed shape: {X_corn_pre.shape}")
print(f"Mean of first spectrum: {X_corn_pre[0].mean():.4f} (should be ~0)")
print(f"Std  of first spectrum: {X_corn_pre[0].std():.4f}  (should be ~1)")

Preprocessed shape: (80, 700)
Mean of first spectrum: 0.0000 (should be ~0)
Std  of first spectrum: 1.0000  (should be ~1)


In [6]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_corn_pre, y_corn, test_size=0.2, random_state=42
)

print(f"Train: {X_tr.shape} — {len(X_tr)} samples")
print(f"Test:  {X_te.shape} — {len(X_te)} samples")

Train: (64, 700) — 64 samples
Test:  (16, 700) — 16 samples


In [7]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Find best components
best_n, best_rmse = 1, np.inf
for n in range(1, 20):
    plsr_cv = PLSRegression(n_components=n)
    scores  = cross_val_score(plsr_cv, X_tr, y_tr,
                              cv=5, scoring='neg_mean_squared_error')
    rmse_cv = np.sqrt(-scores.mean())
    if rmse_cv < best_rmse:
        best_rmse, best_n = rmse_cv, n

plsr = PLSRegression(n_components=best_n)
plsr.fit(X_tr, y_tr)
y_pred = plsr.predict(X_te).flatten()

rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)
rpd  = y_te.std() / rmse

print(f"=== PLSR Results ===")
print(f"Best components: {best_n}")
print(f"RMSE: {rmse:.4f}  R²: {r2:.4f}  RPD: {rpd:.4f}")

=== PLSR Results ===
Best components: 15
RMSE: 0.1287  R²: 0.9258  RPD: 3.6716


In [8]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_te_s   = scaler.transform(X_te)

param_grid = {
    'C':       [0.1, 1, 10, 100],
    'epsilon': [0.001, 0.01, 0.1],
    'gamma':   ['scale', 'auto']
}

svr_gs = GridSearchCV(SVR(kernel='rbf'), param_grid,
                      cv=5, scoring='neg_mean_squared_error',
                      n_jobs=-1, verbose=1)
svr_gs.fit(X_tr_s, y_tr)
y_pred = svr_gs.predict(X_te_s)

rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)
rpd  = y_te.std() / rmse

print(f"\n=== SVR Results ===")
print(f"Best params: {svr_gs.best_params_}")
print(f"RMSE: {rmse:.4f}  R²: {r2:.4f}  RPD: {rpd:.4f}")

Fitting 5 folds for each of 24 candidates, totalling 120 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version


=== SVR Results ===
Best params: {'C': 10, 'epsilon': 0.01, 'gamma': 'auto'}
RMSE: 0.1875  R²: 0.8427  RPD: 2.5212


[Parallel(n_jobs=-1)]: Done 120 out of 120 | elapsed:    1.4s finished


In [9]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_te_s   = scaler.transform(X_te)

param_grid = {
    'C':       [0.1, 1, 10, 100],
    'epsilon': [0.001, 0.01, 0.1],
    'gamma':   ['scale', 'auto']
}

svr_gs = GridSearchCV(SVR(kernel='rbf'), param_grid,
                      cv=5, scoring='neg_mean_squared_error',
                      n_jobs=-1, verbose=1)
svr_gs.fit(X_tr_s, y_tr)
y_pred = svr_gs.predict(X_te_s)

rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)
rpd  = y_te.std() / rmse

print(f"\n=== SVR Results ===")
print(f"Best params: {svr_gs.best_params_}")
print(f"RMSE: {rmse:.4f}  R²: {r2:.4f}  RPD: {rpd:.4f}")

Fitting 5 folds for each of 24 candidates, totalling 120 fits

=== SVR Results ===
Best params: {'C': 10, 'epsilon': 0.01, 'gamma': 'auto'}
RMSE: 0.1875  R²: 0.8427  RPD: 2.5212


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=-1)]: Done 120 out of 120 | elapsed:    0.1s finished


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2']
}

rf_gs = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_dist, n_iter=20, cv=5,
    scoring='neg_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=1
)
rf_gs.fit(X_tr, y_tr)
y_pred = rf_gs.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)
rpd  = y_te.std() / rmse

print(f"\n=== Random Forest Results ===")
print(f"Best params: {rf_gs.best_params_}")
print(f"RMSE: {rmse:.4f}  R²: {r2:.4f}  RPD: {rpd:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 48 concurrent workers.



=== Random Forest Results ===
Best params: {'n_estimators': 100, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': None}
RMSE: 0.3003  R²: 0.5963  RPD: 1.5739


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    1.1s finished


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class CNN1D(nn.Module):
    def __init__(self, input_size=700):
        super(CNN1D, self).__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.flat_size = 128 * (input_size // 16)
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flat_size, 128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv_blocks(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x.squeeze(1)

def train_cnn_fixed(model, X_train, y_train,
                    epochs=100, batch_size=16, lr=1e-4):
    X_tr = torch.FloatTensor(X_train).to(device)
    y_tr = torch.FloatTensor(y_train).to(device)
    dataset  = TensorDataset(X_tr, y_tr)
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    criterion = nn.SmoothL1Loss(beta=0.02)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
                    optimizer, max_lr=lr,
                    steps_per_epoch=len(loader), epochs=epochs)
    model.to(device)
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = nn.SmoothL1Loss(beta=0.02)(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
    return model

torch.manual_seed(42)
model_corn = CNN1D(input_size=700).to(device)
model_corn = train_cnn_fixed(model_corn, X_tr, y_tr, epochs=100)

model_corn.eval()
with torch.no_grad():
    y_pred = model_corn(torch.FloatTensor(X_te).to(device)).cpu().numpy()

rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)
rpd  = y_te.std() / rmse

print(f"\n=== CNN Results ===")
print(f"RMSE: {rmse:.4f}  R²: {r2:.4f}  RPD: {rpd:.4f}")


=== CNN Results ===
RMSE: 0.4821  R²: -0.0403  RPD: 0.9804


In [14]:
# Store all results
results_corn = {
    "PLSR": {"rmse": 0.1287, "r2": 0.9258, "rpd": 3.6716},
    "SVR":  {"rmse": 0.1875, "r2": 0.8427, "rpd": 2.5212},
    "RF":   {"rmse": 0.3003, "r2": 0.5963, "rpd": 1.5739},
    "CNN":  {"rmse": 0.4821, "r2": -0.0403, "rpd": 0.9804},
}

print("\n" + "="*55)
print("   CORN DATASET — ALL MODELS COMPARISON")
print("="*55)
print(f"{'Model':<15} {'RMSE':>8} {'R²':>8} {'RPD':>8}")
print("-"*55)
for model, res in results_corn.items():
    print(f"{model:<15} {res['rmse']:>8.4f} {res['r2']:>8.4f} {res['rpd']:>8.4f}")
print("="*55)

print("\n=== VALIDATION AGAINST PUBLISHED RESULTS ===")
print(f"PLSR R²   = {results_corn['PLSR']['r2']:.4f}  (expected 0.90 to 0.95) ✅ MATCH")
print(f"PLSR RMSE = {results_corn['PLSR']['rmse']:.4f}  (expected 0.10 to 0.20) ✅ MATCH")

print("\n=== CONCLUSION ===")
print("PLSR and SVR code confirmed correct.")
print("CNN fails on corn — expected with only 64 training samples.")
print("Code is NOT buggy. Challenge is data size for CNN.")


   CORN DATASET — ALL MODELS COMPARISON
Model               RMSE       R²      RPD
-------------------------------------------------------
PLSR              0.1287   0.9258   3.6716
SVR               0.1875   0.8427   2.5212
RF                0.3003   0.5963   1.5739
CNN               0.4821  -0.0403   0.9804

=== VALIDATION AGAINST PUBLISHED RESULTS ===
PLSR R²   = 0.9258  (expected 0.90 to 0.95) ✅ MATCH
PLSR RMSE = 0.1287  (expected 0.10 to 0.20) ✅ MATCH

=== CONCLUSION ===
PLSR and SVR code confirmed correct.
CNN fails on corn — expected with only 64 training samples.
Code is NOT buggy. Challenge is data size for CNN.
